# 🔁 Notebook 1: Retry with Exponential Backoff & Jitter

**The setup:** A flaky downstream service fails ~30% of the time. Naive code retries instantly and *immediately* — making the outage worse (a 'thundering herd').

We'll evolve from naive → exponential backoff → backoff with jitter.


## 🛠️ Setup

```bash
cd 04-patterns/resilience
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: tight retry loop

In [ ]:
import random, time
random.seed(0)

def flaky_call(failure_rate=0.7):
    if random.random() < failure_rate:
        raise IOError('downstream is sad')
    return 'ok'

def retry_naive(fn, attempts=8):
    delays = []
    for i in range(attempts):
        try:
            return fn(), delays
        except IOError:
            delays.append(0)  # instant retry
    raise IOError('giving up')

try:
    result, delays = retry_naive(flaky_call)
    print('got', result, 'after', len(delays), 'retries — total wait', sum(delays), 's')
except IOError as e:
    print('failed:', e)


All clients hammer at full speed = the outage gets *worse*.

## 🟨 BETTER: exponential backoff

In [ ]:
def retry_exp(fn, attempts=8, base=0.05, cap=2.0):
    delays = []
    for i in range(attempts):
        try: return fn(), delays
        except IOError:
            wait = min(cap, base * 2**i)
            delays.append(wait)
            time.sleep(wait)
    raise IOError('giving up')

random.seed(0)
result, delays = retry_exp(flaky_call)
print('delays:', [round(d,3) for d in delays])


Now each retry waits longer — but **every client backs off in lockstep**, syncing up the herd. Add **jitter**.

## 🟩 BEST: full-jitter exponential backoff (AWS recipe)

In [ ]:
def retry_jitter(fn, attempts=8, base=0.05, cap=2.0):
    delays = []
    for i in range(attempts):
        try: return fn(), delays
        except IOError:
            ceiling = min(cap, base * 2**i)
            wait = random.uniform(0, ceiling)  # full jitter
            delays.append(wait)
            time.sleep(wait)
    raise IOError('giving up')

random.seed(0)
result, delays = retry_jitter(flaky_call)
print('delays:', [round(d,3) for d in delays])


## 🧠 Summary

| Strategy | Pros | Cons |
|---|---|---|
| Tight loop | quickest success on transient blips | thundering herd |
| Exponential | bounded total wait | clients sync up |
| **Exponential + full jitter** | spreads load, recommended | slightly more code |

Always cap retries (and total wait) — never retry forever.